In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv


In [2]:
import numpy as np
import pandas as pd

In [3]:
#load 
train= pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv')
test= pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv')

print(train.shape)
print(test.shape)

train.head(2)

(1460, 81)
(1459, 80)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500


In [4]:
#column with highest missing value
print(train.isnull().sum().sort_values(ascending=False).head())

PoolQC         1453
MiscFeature    1406
Alley          1369
Fence          1179
MasVnrType      872
dtype: int64


In [5]:
cols_to_drop = ['PoolQC', 'MiscFeature', 'Alley', 'Fence']

train = train.drop(cols_to_drop, axis=1)
test = test.drop(cols_to_drop, axis=1)

In [6]:
#check diffrence between values 
"""
0       → Symmetric
0.5     → Slightly skewed
1.0+    → Highly skewed
1.88    → Very highly right-skewed ✅
"""
train['SalePrice'].skew()

np.float64(1.8828757597682129)

In [7]:
X = train.drop('SalePrice', axis=1)
y = train['SalePrice']

y=np.log1p(y)

print(y.head())

0    12.247699
1    12.109016
2    12.317171
3    11.849405
4    12.429220
Name: SalePrice, dtype: float64


In [8]:
#Apply V2 Missing Value Handling
num_cols = X.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns

X[num_cols] = X[num_cols].fillna(X[num_cols].median())
test[num_cols] = test[num_cols].fillna(X[num_cols].median())

X[cat_cols] = X[cat_cols].fillna('Missing')
test[cat_cols] = test[cat_cols].fillna('Missing')

#Verify missing value
print(X.isnull().sum().sum())
print(test.isnull().sum().sum())

0
0


In [9]:
#One-Hot Encode
X = pd.get_dummies(X)
test_encoded = pd.get_dummies(test)

X, test_encoded = X.align(
    test_encoded,
    join='left',
    axis=1,
    fill_value=0
)

print(X.shape)
print(test_encoded.shape)

(1460, 287)
(1459, 287)


# Train Random Forest on Log Prices

In [10]:
from sklearn.ensemble import RandomForestRegressor

model=RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X,y)

RandomForestRegressor(random_state=42)

log1p()  → compress,

expm1()  → restore

In [11]:
prediction = model.predict(test_encoded)

#Convert Back to Real Prices
prediction = np.expm1(prediction)
print(prediction)

[123331.86854736 154040.76973929 179761.13688947 ... 152288.38265395
 111536.96043322 234006.6784248 ]


In [12]:
print(prediction[:10])

[123331.86854736 154040.76973929 179761.13688947 182630.30042451
 194874.26074733 183195.53952185 159954.39199406 176804.15324913
 182442.93398933 121659.49533148]


In [13]:
#create submission file
submission=pd.DataFrame({
    'Id':test['Id'],
    'SalePrice': prediction
})

submission.to_csv('submission_v3.csv',index=False)

submission.head()

,Id,SalePrice
0,1461,123331.868547
1,1462,154040.769739
2,1463,179761.136889
3,1464,182630.300425
4,1465,194874.260747
